# Prompt alignment + CLAP

In [1]:
import scipy
import torch
from diffusers import AudioLDM2Pipeline
from IPython.display import Audio
import pandas as pd
import laion_clap
import glob
import json
import torch
import numpy as np
import scipy.io.wavfile as wav
from pathlib import Path


In [2]:
def reinit_pipe():
    repo_id = "cvssp/audioldm2-large"
    pipe = AudioLDM2Pipeline.from_pretrained(repo_id, torch_dtype=torch.float16)
    pipe = pipe.to("cuda")
    return pipe

In [ ]:
repo_id = "cvssp/audioldm2-large"
pipe = AudioLDM2Pipeline.from_pretrained(repo_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")

In [ ]:
clap_device = torch.device('cuda:0')
clap_model = laion_clap.CLAP_Module(enable_fusion=True, device=clap_device)
clap_model.load_ckpt(verbose=False)


In [5]:
def save_audios(audios: list, dir_name: str):
    Path(dir_name).mkdir(parents=True, exist_ok=True)
    for i, audio in enumerate(audios):
        wav.write(f"{dir_name}/a_id{i}.wav", 16000, audio)

In [6]:
def gen_latents(pipe, n_prompts, n_latents_per_prompt=1, seed=0):
    NUM_CHANNEL_LATENTS = 8
    HEIGHT = 1024
    DTYPE=torch.float16
    DEVICE=torch.device("cuda")
    GENERATOR = torch.Generator("cuda").manual_seed(seed)
    single_prompt_latent = pipe.prepare_latents(
        n_latents_per_prompt,
        NUM_CHANNEL_LATENTS,
        HEIGHT,
        DTYPE,
        DEVICE,
        GENERATOR,
        None
    )
    return single_prompt_latent.repeat(n_prompts, 1, 1, 1)

In [7]:
def calc_clap_metrics(clap_model, classes: list, audio_dirs: dict, clap_prompt: str = "This is a music of "):
    class_index_dict = {
        class_name: i for i, class_name in enumerate(classes)
    }
    audio_files = []
    gts = []
    for class_name, audio_dir in audio_dirs.items():
        paths = sorted(Path(audio_dir).glob("*.wav"))
        audio_files.extend(paths)
        gts.extend([class_index_dict[class_name] for _ in range(len(paths))])

    audio_files = [str(p) for p in audio_files]
    gts = torch.tensor(gts)

    clap_model = clap_model.eval()

    with torch.no_grad():
        all_texts = [clap_prompt + t for t in class_index_dict.keys()]
        text_embed = clap_model.get_text_embedding(all_texts)
        audio_embed = clap_model.get_audio_embedding_from_filelist(x=audio_files)

        sims = torch.tensor(audio_embed) @ torch.tensor(text_embed).t()

        ranking = torch.argsort(sims, descending=True)
        ranking_preds = ranking[:, 0]

        preds = torch.where(ranking_preds == gts, 1, 0)
        preds = preds.cpu().numpy()

        metrics = {}
        metrics["accuracy"] = preds.mean()
        metrics["precision"] = preds.sum() / len(preds)
        metrics["recall"] = preds.sum() / len(preds)
        metrics["f1"] = 2 * (metrics["precision"] * metrics["recall"]) / (metrics["precision"] + metrics["recall"])
        print(
            f"Zeroshot Classification Results: "
            + "\t".join([f"{k}: {round(v, 4):.4f}" for k, v in metrics.items()])
        )
        print(f"Mean similarities per class: {sims.mean(dim=0)}")
        print("Similarity matrix:")
        print(sims)

In [8]:
df = pd.read_csv("data/generated_prompts.csv")

In [9]:
NEGATIVE_PROMPT = "Low quality, average quality."
NUM_INFERENCE_STEPS = 400
NUM_WAVEFORMS_PER_PROMPT = 1
GUIDANCE_SCALE = 5
SEED = 222
N_LATENTS_PER_PROMPT = 4
AUDIO_LEN_IN_S = 9

# Female and Male

In [10]:
df_male_female = df[df["original_feature"].isin(["male", "female"])]

In [11]:
prompts_female = df_male_female["original_prompt"].tolist()
prompts_female = ["Female woman singing a song. " + pt + ". Female woman voice." for pt in prompts_female]
prompts_male = df_male_female["modified_prompt"].tolist()
prompts_male = ["Male man singing a song. " + pt + ". Male man voice." for pt in prompts_male]

In [12]:
latents_female = gen_latents(pipe=pipe, n_prompts=len(prompts_female), n_latents_per_prompt=N_LATENTS_PER_PROMPT, seed=SEED)
prompts_female_repeated = [pt for pt in prompts_female for _ in range(N_LATENTS_PER_PROMPT)]
neg_prompts_female = [NEGATIVE_PROMPT for _ in range(len(prompts_female_repeated))]

In [ ]:
generator = torch.Generator("cuda").manual_seed(SEED)
audios = pipe(
    prompts_female_repeated,
    negative_prompt=neg_prompts_female,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    latents=latents_female,
    audio_length_in_s=AUDIO_LEN_IN_S,
).audios

In [14]:
save_audios(audios, "outputs/female")

In [ ]:
calc_clap_metrics(clap_model, classes=["female voice singing", "male voice singing"], audio_dirs={"female voice singing": "outputs/female"})

In [ ]:
prompts_male_repeated = [pt for pt in prompts_male for _ in range(N_LATENTS_PER_PROMPT)]
neg_prompts_male = [NEGATIVE_PROMPT for _ in range(len(prompts_male_repeated))]
generator = torch.Generator("cuda").manual_seed(SEED)
audios = pipe(
    prompts_male_repeated,
    negative_prompt=neg_prompts_male,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    latents=latents_female,
    audio_length_in_s=AUDIO_LEN_IN_S,
).audios

In [17]:
save_audios(audios, "outputs/male")

In [ ]:
calc_clap_metrics(clap_model, classes=["female voice singing", "male voice singing"], audio_dirs={"male voice singing": "outputs/male"})



# Slow and Fast

In [ ]:
repo_id = "cvssp/audioldm2-large"
pipe = AudioLDM2Pipeline.from_pretrained(repo_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")

In [48]:
df_slow_fast = df[df["original_feature"].isin(["slow", "fast"])]

In [49]:
prompts_slow = df_slow_fast["original_prompt"].tolist()
prompts_slow = ["Very slow song. " + pt + ". Slow song." for pt in prompts_slow]
prompts_fast = df_slow_fast["modified_prompt"].tolist()
prompts_fast = ["Very fast song. " + pt + ". Fast song." for pt in prompts_fast]

In [ ]:
latents_slow = gen_latents(pipe=pipe, n_prompts=len(prompts_slow), n_latents_per_prompt=N_LATENTS_PER_PROMPT, seed=SEED)
prompts_slow_repeated = [pt for pt in prompts_slow for _ in range(N_LATENTS_PER_PROMPT)]
neg_prompts_slow = [NEGATIVE_PROMPT for _ in range(len(prompts_slow_repeated))]
generator = torch.Generator("cuda").manual_seed(SEED)
audios = pipe(
    prompts_slow_repeated,
    negative_prompt=neg_prompts_slow,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    latents=latents_slow,
    audio_length_in_s=AUDIO_LEN_IN_S,
).audios
save_audios(audios, "outputs/slow")

In [ ]:
calc_clap_metrics(clap_model, classes=["slow song", "fast song"], audio_dirs={"slow song": "outputs/slow"})

In [ ]:
latents_fast = gen_latents(pipe=pipe, n_prompts=len(prompts_fast), n_latents_per_prompt=N_LATENTS_PER_PROMPT, seed=SEED)
prompts_fast_repeated = [pt for pt in prompts_fast for _ in range(N_LATENTS_PER_PROMPT)]
neg_prompts_fast = [NEGATIVE_PROMPT for _ in range(len(prompts_fast_repeated))]
generator = torch.Generator("cuda").manual_seed(SEED)
audios = pipe(
    prompts_fast_repeated,
    negative_prompt=neg_prompts_fast,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    latents=latents_fast,
    audio_length_in_s=AUDIO_LEN_IN_S,
).audios
save_audios(audios, "outputs/fast")

In [ ]:
calc_clap_metrics(clap_model, classes=["slow music", "fast music"], audio_dirs={"fast music": "outputs/fast"})



# happy and sad

In [20]:

df_happy_sad = df[df["original_feature"].isin(["happy", "sad"])]
prompts_happy = df_happy_sad["original_prompt"].tolist()
prompts_happy = ["Happy song. " + pt + ". Music with a happy vibe." for pt in prompts_happy]
prompts_sad = df_happy_sad["modified_prompt"].tolist()
prompts_sad = ["Sad song. " + pt + ". Music with a sad vibe." for pt in prompts_sad]


In [ ]:
latents_happy = gen_latents(pipe=pipe, n_prompts=len(prompts_happy), n_latents_per_prompt=N_LATENTS_PER_PROMPT, seed=SEED)
prompts_happy_repeated = [pt for pt in prompts_happy for _ in range(N_LATENTS_PER_PROMPT)]
neg_prompts_happy = [NEGATIVE_PROMPT for _ in range(len(prompts_happy_repeated))]
generator = torch.Generator("cuda").manual_seed(SEED)
audios = pipe(
    prompts_happy_repeated,
    negative_prompt=neg_prompts_happy,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    latents=latents_happy,
    audio_length_in_s=AUDIO_LEN_IN_S,
).audios

In [22]:

save_audios(audios, "outputs/happy")

In [ ]:
calc_clap_metrics(clap_model, classes=["happy music", "sad music"], audio_dirs={"happy music": "outputs/happy"})



In [ ]:
pipe = reinit_pipe()
latents_sad = gen_latents(pipe=pipe, n_prompts=len(prompts_sad), n_latents_per_prompt=N_LATENTS_PER_PROMPT, seed=SEED)
prompts_sad_repeated = [pt for pt in prompts_sad for _ in range(N_LATENTS_PER_PROMPT)]
neg_prompts_sad = [NEGATIVE_PROMPT for _ in range(len(prompts_sad_repeated))]
generator = torch.Generator("cuda").manual_seed(SEED)
audios = pipe(
    prompts_sad_repeated,
    negative_prompt=neg_prompts_sad,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    latents=latents_sad,
    audio_length_in_s=AUDIO_LEN_IN_S,
).audios
save_audios(audios, "outputs/sad")

In [ ]:
calc_clap_metrics(clap_model, classes=["happy music", "sad music"], audio_dirs={"sad music": "outputs/sad"})


# Reggae and Classic

In [18]:

df_reggae_classic = df[df["original_feature"].isin(["reggae", "classic"])]
prompts_reggae = df_reggae_classic["original_prompt"].tolist()
# prompts_happy = ["Happy song. " + pt + ". Happy song." for pt in prompts_happy]
prompts_classic = df_reggae_classic["modified_prompt"].tolist()
# prompts_sad = ["Sad song. " + pt + ". Sad song." for pt in prompts_sad]

In [ ]:
pipe = reinit_pipe()
latents_reggae = gen_latents(pipe=pipe, n_prompts=len(prompts_reggae), n_latents_per_prompt=N_LATENTS_PER_PROMPT, seed=SEED)
prompts_reggae_repeated = [pt for pt in prompts_reggae for _ in range(N_LATENTS_PER_PROMPT)]
neg_prompts_reggae = [NEGATIVE_PROMPT for _ in range(len(prompts_reggae_repeated))]
generator = torch.Generator("cuda").manual_seed(SEED)
audios = pipe(
    prompts_reggae_repeated,
    negative_prompt=neg_prompts_reggae,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    latents=latents_reggae,
    audio_length_in_s=AUDIO_LEN_IN_S,
).audios
save_audios(audios, "outputs/reggae")


In [ ]:
calc_clap_metrics(clap_model, classes=["reggae music", "classic music"], audio_dirs={"reggae music": "outputs/reggae"})


In [ ]:
pipe = reinit_pipe()
latents_classic = gen_latents(pipe=pipe, n_prompts=len(prompts_classic), n_latents_per_prompt=N_LATENTS_PER_PROMPT, seed=SEED)
prompts_classic_repeated = [pt for pt in prompts_classic for _ in range(N_LATENTS_PER_PROMPT)]
neg_prompts_classic = [NEGATIVE_PROMPT for _ in range(len(prompts_classic_repeated))]
generator = torch.Generator("cuda").manual_seed(SEED)
audios = pipe(
    prompts_classic_repeated,
    negative_prompt=neg_prompts_classic,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    latents=latents_classic,
    audio_length_in_s=AUDIO_LEN_IN_S,
).audios
save_audios(audios, "outputs/classic")


In [ ]:
calc_clap_metrics(clap_model, classes=["reggae music", "classic music"], audio_dirs={"classic music": "outputs/classic"})


# Violin and Trumpet

In [30]:

df_violin_trumpet = df[df["original_feature"].isin(["violin", "trumpet"])]
prompts_violin = df_violin_trumpet["original_prompt"].tolist()
# prompts_happy = ["Happy song. " + pt + ". Happy song." for pt in prompts_happy]
prompts_trumpet = df_violin_trumpet["modified_prompt"].tolist()
# prompts_sad = ["Sad song. " + pt + ". Sad song." for pt in prompts_sad]

In [ ]:
pipe = reinit_pipe()
latents_violin = gen_latents(pipe=pipe, n_prompts=len(prompts_violin), n_latents_per_prompt=N_LATENTS_PER_PROMPT, seed=SEED)
prompts_violin_repeated = [pt for pt in prompts_violin for _ in range(N_LATENTS_PER_PROMPT)]
neg_prompts_violin = [NEGATIVE_PROMPT for _ in range(len(prompts_violin_repeated))]
generator = torch.Generator("cuda").manual_seed(SEED)
audios = pipe(
    prompts_violin_repeated,
    negative_prompt=neg_prompts_violin,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    latents=latents_violin,
    audio_length_in_s=AUDIO_LEN_IN_S,
).audios
save_audios(audios, "outputs/violin")

calc_clap_metrics(clap_model, classes=["violin music", "trumpet music"], audio_dirs={"violin music": "outputs/violin"})

In [ ]:
pipe = reinit_pipe()
latents_trumpet = gen_latents(pipe=pipe, n_prompts=len(prompts_trumpet), n_latents_per_prompt=N_LATENTS_PER_PROMPT, seed=SEED)
prompts_trumpet_repeated = [pt for pt in prompts_trumpet for _ in range(N_LATENTS_PER_PROMPT)]
neg_prompts_trumpet = [NEGATIVE_PROMPT for _ in range(len(prompts_trumpet_repeated))]
generator = torch.Generator("cuda").manual_seed(SEED)
audios = pipe(
    prompts_trumpet_repeated,
    negative_prompt=neg_prompts_trumpet,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    latents=latents_trumpet,
    audio_length_in_s=AUDIO_LEN_IN_S,
).audios
save_audios(audios, "outputs/trumpet")

calc_clap_metrics(clap_model, classes=["violin music", "trumpet music"], audio_dirs={"trumpet music": "outputs/trumpet"})

# Saxophone and Drums

In [33]:

df_saxophone_drums = df[df["original_feature"].isin(["saxophone", "drums"])]
prompts_saxophone = df_saxophone_drums["original_prompt"].tolist()
# prompts_happy = ["Happy song. " + pt + ". Happy song." for pt in prompts_happy]
prompts_drums = df_saxophone_drums["modified_prompt"].tolist()
# prompts_sad = ["Sad song. " + pt + ". Sad song." for pt in prompts_sad]

In [ ]:
pipe = reinit_pipe()
latents_saxophone = gen_latents(pipe=pipe, n_prompts=len(prompts_saxophone), n_latents_per_prompt=N_LATENTS_PER_PROMPT, seed=SEED)
prompts_saxophone_repeated = [pt for pt in prompts_saxophone for _ in range(N_LATENTS_PER_PROMPT)]
neg_prompts_saxophone = [NEGATIVE_PROMPT for _ in range(len(prompts_saxophone_repeated))]
generator = torch.Generator("cuda").manual_seed(SEED)
audios = pipe(
    prompts_saxophone_repeated,
    negative_prompt=neg_prompts_saxophone,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    latents=latents_saxophone,
    audio_length_in_s=AUDIO_LEN_IN_S,
).audios
save_audios(audios, "outputs/saxophone")

calc_clap_metrics(clap_model, classes=["saxophone music", "drums music"], audio_dirs={"saxophone music": "outputs/saxophone"})

In [ ]:
pipe = reinit_pipe()
latents_drums = gen_latents(pipe=pipe, n_prompts=len(prompts_drums), n_latents_per_prompt=N_LATENTS_PER_PROMPT, seed=SEED)
prompts_drums_repeated = [pt for pt in prompts_drums for _ in range(N_LATENTS_PER_PROMPT)]
neg_prompts_drums = [NEGATIVE_PROMPT for _ in range(len(prompts_drums_repeated))]
generator = torch.Generator("cuda").manual_seed(SEED)
audios = pipe(
    prompts_drums_repeated,
    negative_prompt=neg_prompts_drums,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    latents=latents_drums,
    audio_length_in_s=AUDIO_LEN_IN_S,
).audios
save_audios(audios, "outputs/drums")

calc_clap_metrics(clap_model, classes=["saxophone music", "drums music"], audio_dirs={"drums music": "outputs/drums"})


# Soft and Loud

In [36]:
df_soft_loud = df[df["original_feature"].isin(["soft", "loud"])]
prompts_soft = df_soft_loud["original_prompt"].tolist()
# prompts_happy = ["Happy song. " + pt + ". Happy song." for pt in prompts_happy]
prompts_loud = df_soft_loud["modified_prompt"].tolist()
# prompts_sad = ["Sad song. " + pt + ". Sad song." for pt in prompts_sad]

In [ ]:
pipe = reinit_pipe()
latents_soft = gen_latents(pipe=pipe, n_prompts=len(prompts_soft), n_latents_per_prompt=N_LATENTS_PER_PROMPT, seed=SEED)
prompts_soft_repeated = [pt for pt in prompts_soft for _ in range(N_LATENTS_PER_PROMPT)]
neg_prompts_soft = [NEGATIVE_PROMPT for _ in range(len(prompts_soft_repeated))]
generator = torch.Generator("cuda").manual_seed(SEED)
audios = pipe(
    prompts_soft_repeated,
    negative_prompt=neg_prompts_soft,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    latents=latents_soft,
    audio_length_in_s=AUDIO_LEN_IN_S,
).audios
save_audios(audios, "outputs/soft")

calc_clap_metrics(clap_model, classes=["soft music", "loud music"], audio_dirs={"soft music": "outputs/soft"})

In [ ]:
pipe = reinit_pipe()
latents_loud = gen_latents(pipe=pipe, n_prompts=len(prompts_loud), n_latents_per_prompt=N_LATENTS_PER_PROMPT, seed=SEED)
prompts_loud_repeated = [pt for pt in prompts_loud for _ in range(N_LATENTS_PER_PROMPT)]
neg_prompts_loud = [NEGATIVE_PROMPT for _ in range(len(prompts_loud_repeated))]
generator = torch.Generator("cuda").manual_seed(SEED)
audios = pipe(
    prompts_loud_repeated,
    negative_prompt=neg_prompts_loud,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    latents=latents_loud,
    audio_length_in_s=AUDIO_LEN_IN_S,
).audios
save_audios(audios, "outputs/loud")

calc_clap_metrics(clap_model, classes=["soft music", "loud music"], audio_dirs={"loud music": "outputs/loud"})
